# 01.05 · Streaming, batching, and async basics

**Module 01 · Models, Messages & Prompts** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 01.01–01.04. Python functions and loops; no async experience needed. |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (fake model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Stream** tokens to the notebook, separate reasoning from answer text, and **aggregate** `AIMessageChunk`s into a full message (`+`, `chunk_position`).
2. **Measure** time-to-first-token vs total time, and explain why streaming improves perceived speed but not total speed.
3. **Render** a live-updating Markdown answer with an IPython display handle (rich display).
4. **Batch** requests with `max_concurrency`, keep going when one fails (`return_exceptions=True`), and process results as they complete.
5. **Write** your first async code: top-level `await` in Jupyter, `ainvoke`, `astream`, `abatch`, `asyncio.gather` with a semaphore, and `astream_events`.

## 📚 Official reading (verified 2026-10-11)

| Page | Read it for |
|---|---|
| [Models → Invocation (invoke, stream, batch)](https://docs.langchain.com/oss/python/langchain/models#invocation) | **Main page.** Chunk aggregation, streaming events, `batch_as_completed`, `max_concurrency` |
| [Streaming](https://docs.langchain.com/oss/python/langchain/streaming) | How agents stream (stream modes, `chunk_position`); you'll use it in Module 02 |
| [Python `asyncio`](https://docs.python.org/3/library/asyncio.html) · [Coroutines and tasks](https://docs.python.org/3/library/asyncio-task.html) | `async`/`await`, `gather`, `Semaphore` |
| [IPython display](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html) | `display(..., display_id=True)` and `handle.update(...)` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/01-models-messages-prompts/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · What a stream really is

`stream()` returns an **iterator** of `AIMessageChunk` objects. Each chunk carries a small piece: some text, a slice of reasoning, part of a tool call's arguments. Chunks **add up**: `chunk_a + chunk_b` is a bigger chunk, so summing them all rebuilds the full message, including `usage_metadata` (usually attached to the last chunk). The last chunk is marked `chunk_position == "last"`.

Let's see the mechanics offline with the fake model from Lesson 01.01.

In [ ]:
# 🧪 Offline: stream from a fake model, aggregate the chunks, inspect the result.
from langchain.messages import AIMessage, AIMessageChunk
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

fake = GenericFakeChatModel(messages=iter([AIMessage("Streaming lets users read while the model writes.")]))
full: AIMessageChunk | None = None
for i, chunk in enumerate(fake.stream("Why stream?")):
    full = chunk if full is None else full + chunk      # + merges text, reasoning, tool-call pieces, usage
    if i < 4 or chunk.chunk_position == "last":
        print(f"chunk {i:>2}: {chunk.text!r:<12} position={chunk.chunk_position}")
print("\nAggregated type:", type(full).__name__)
print("Aggregated text:", full.text)

### What just happened?

- Each chunk held one word or one space; only the final one had `position='last'`.
- `full` is an `AIMessageChunk` with the whole text. You can append it to a conversation exactly like an `AIMessage` from `invoke` ([Models → Stream](https://docs.langchain.com/oss/python/langchain/models#stream)).
- **Why `chunk_position` matters:** in a long stream (an agent that answers, calls a tool, answers again), `"last"` tells you a message is complete, so you can act on it (Module 02).

---
## Part 2 · Real streaming: reasoning first, then the answer

`gpt-oss` models reason before answering. In a stream, reasoning arrives as `reasoning` content blocks, then the answer as `text` blocks (Lesson 01.02). We'll print them differently and measure **time to first token** (TTFT): how long until *anything* appears. That's what users feel.

In [ ]:
# 💸 Calls Groq (fast model), ~400 tokens. Reasoning in brackets, then the answer.
import time

from mastery.models import get_model

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    llm = get_model("fast", temperature=0, max_tokens=600, reasoning_effort="low")
    start, first_token_at, full = time.perf_counter(), None, None
    for chunk in llm.stream("Explain in 3 short bullet points why streaming improves UX."):
        first_token_at = first_token_at or time.perf_counter()
        full = chunk if full is None else full + chunk
        for block in chunk.content_blocks:
            if block["type"] == "reasoning":
                print(f"[{block.get('reasoning', '')}]", end="", flush=True)
            elif block["type"] == "text":
                print(block["text"], end="", flush=True)
    total = time.perf_counter() - start
    print(f"\n\nTTFT {first_token_at - start:.2f}s · total {total:.2f}s · usage {full.usage_metadata}")

### What just happened?

- Text in `[brackets]` was reasoning; it came first. Then the bullets.
- **TTFT is a fraction of the total.** Streaming doesn't make the model faster; it lets people start reading sooner. For a 10-second answer, that's the difference between "is it broken?" and "it's working".
- `full.usage_metadata` has the token counts because they were attached to the final chunk and aggregation kept them. Not every API does this: OpenAI's Chat Completions API only streams usage when asked. `ChatOpenAI` (`langchain-openai` 1.7.0) asks **by default** when it talks to OpenAI's own URL, but leaves it off if you set `OPENAI_BASE_URL` to another endpoint; force it with `stream_usage=True` ([ChatOpenAI → Streaming usage metadata](https://docs.langchain.com/oss/python/integrations/chat/openai#streaming-usage-metadata); default checked in the package source on 2026-10-11).
- In a product you usually **hide** reasoning or show a short "thinking..." indicator. Here we print it because you're learning.

---
## Part 3 · Rich display: a live-updating Markdown answer

Printing chunks gives raw text. Models write Markdown (bullets, bold, code), so let's render it **as it streams**. The trick: `display(obj, display_id=True)` returns a **handle**; `handle.update(new_obj)` replaces that output in place. Updating on every token is wasteful, so we refresh every few chunks.

In [ ]:
# 🧪 Offline: a reusable live-Markdown renderer for any stream of chunks.
from IPython.display import Markdown, display


def stream_to_markdown(chunks, refresh_every: int = 5):
    """Render a chunk stream as live-updating Markdown; return the aggregated message."""
    handle = display(Markdown("⏳ ..."), display_id=True)
    full = None
    for i, chunk in enumerate(chunks, start=1):
        full = chunk if full is None else full + chunk
        if i % refresh_every == 0:
            handle.update(Markdown(full.text + " ▌"))   # The block cursor shows it's still writing.
    handle.update(Markdown(full.text if full else "(empty)"))
    return full


fake = GenericFakeChatModel(messages=iter([AIMessage("**Streaming** in a notebook:\n\n- shows progress\n- renders Markdown\n- ends clean")]))
_ = stream_to_markdown(fake.stream("demo"), refresh_every=2)

In [ ]:
# 💸 Calls Groq (fast model), ~500 tokens. Same renderer, real model.
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    answer = stream_to_markdown(llm.stream("Give a 4-row Markdown table: streaming vs batch vs invoke, with 'best for'."))

### What just happened?

You saw formatted Markdown (bold, bullets; with a key, a real table) grow in place instead of a wall of raw characters. Only the **text** is rendered, so reasoning stays hidden. `stream_to_markdown` returns the full message, so nothing is lost. Note the renderer works with **any** chunk stream: fake or real model, Groq or another provider.

> ⚠️ **Production note: streaming in notebooks vs products.**
> `handle.update` only exists in Jupyter. In production, stream to clients with Server-Sent Events or WebSockets (Module 12 uses FastAPI), handle client disconnects (stop generating to save tokens), and sanitize Markdown before rendering it as HTML in a browser.

---
## Part 4 · Batching: parallel requests, safely

`batch()` runs independent inputs in parallel **threads** and returns results in input order. Three options matter:

| Option | Effect |
|---|---|
| `config={"max_concurrency": n}` | At most `n` requests in flight. Your main lever against rate limits |
| `return_exceptions=True` | A failed input returns its exception **in its slot**; the others still succeed |
| `batch_as_completed(...)` | Yields `(index, result)` as soon as each finishes (out of order) |

To *see* concurrency without spending tokens, we use a fake "model" that takes 0.5 s per call: a `RunnableLambda`, the simplest custom `Runnable` (Lesson 01.07).

In [ ]:
# 🧪 Offline: a slow fake model (0.5 s per call). Fails on purpose for the input "boom".
import time

from langchain_core.runnables import RunnableLambda

DELAY = 0.5

def slow_answer(question: str) -> str:
    time.sleep(DELAY)          # Stands in for network latency.
    if question == "boom":
        raise ValueError("simulated provider error")
    return f"answer to {question!r}"

slow_model = RunnableLambda(slow_answer)
QUESTIONS = [f"q{i}" for i in range(1, 7)]

In [ ]:
%%time
# 🧪 Sequential: 6 calls x 0.5 s, about 3 s.
sequential = [slow_model.invoke(q) for q in QUESTIONS]

In [ ]:
%%time
# 🧪 Batch with max_concurrency=3: two "waves" of 3, about 1 s.
batched = slow_model.batch(QUESTIONS, config={"max_concurrency": 3})
print(batched[:2], "...")

In [ ]:
# 🧪 One input fails: with return_exceptions=True the rest of the batch survives.
mixed = slow_model.batch(["q1", "boom", "q3"], config={"max_concurrency": 3}, return_exceptions=True)
for question, result in zip(["q1", "boom", "q3"], mixed):
    status = f"❌ {type(result).__name__}: {result}" if isinstance(result, Exception) else f"✅ {result}"
    print(f"{question:<5} {status}")

In [ ]:
# 🧪 Results as they complete: (index, result) pairs, possibly out of order.
for index, result in slow_model.batch_as_completed(QUESTIONS[:4], config={"max_concurrency": 4}):
    print(index, result)

### What just happened?

- **Sequential ≈ 3 s, batched ≈ 1 s** (`Wall time` lines). With `max_concurrency=6` it would be ≈ 0.5 s. Network-bound work parallelizes well because threads spend their time waiting.
- With `return_exceptions=True`, `boom` became a `ValueError` **in its slot**; `q1` and `q3` succeeded. Without it, the whole `batch()` call raises and you lose the successful results too.
- `batch_as_completed` gave `(index, result)` pairs; use the index to match results to inputs.

---
## Part 5 · Async basics: many requests, one thread

**Why async?** A thread per request works for a notebook, but servers handle thousands of concurrent requests. With `asyncio`, one thread switches between tasks whenever one is **waiting** (for the network). Rules of thumb:

- `async def` defines a **coroutine function**; calling it creates a coroutine; `await` runs it and waits for its result.
- Every LangChain `Runnable` has async twins: `ainvoke`, `astream`, `abatch`, `astream_events`.
- **Jupyter already runs an event loop**, so you can write `await ...` at the top level of a cell. In a `.py` script you'd wrap it in `asyncio.run(main())`. **Don't** call `asyncio.run()` inside Jupyter: it fails with *"cannot be called from a running event loop"*.

Async version of the slow fake: `RunnableLambda(func, afunc=...)` takes both a sync and an async implementation.

In [ ]:
# 🧪 Offline: an async slow model. asyncio.sleep yields to other tasks; time.sleep would block them all.
import asyncio

async def slow_answer_async(question: str) -> str:
    await asyncio.sleep(DELAY)
    return f"answer to {question!r}"

async_model = RunnableLambda(slow_answer, afunc=slow_answer_async)
print(await async_model.ainvoke("q1"))      # Top-level await: works in Jupyter only.

> ⏱️ **`%%time` can't time top-level `await`** (you get `SyntaxError: 'await' outside function`). For async cells, time with `time.perf_counter()` yourself.

In [ ]:
# 🧪 asyncio.gather runs coroutines concurrently. A Semaphore caps concurrency (like max_concurrency).
LIMIT = asyncio.Semaphore(3)

async def ask_limited(question: str) -> str:
    async with LIMIT:                       # At most 3 inside this block at once.
        return await async_model.ainvoke(question)

start = time.perf_counter()
results = await asyncio.gather(*(ask_limited(q) for q in QUESTIONS))
print(f"{len(results)} answers in {time.perf_counter() - start:.2f}s")

In [ ]:
# 🧪 The built-in shortcut: abatch with max_concurrency does the same thing.
start = time.perf_counter()
results = await async_model.abatch(QUESTIONS, config={"max_concurrency": 3})
print(f"{results[-1]}  ({time.perf_counter() - start:.2f}s)")

In [ ]:
# 💸 Calls Groq 3 times concurrently + 1 stream (fast model), ~800 tokens.
if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    replies = await llm.abatch(["Define latency in 8 words.", "Define throughput in 8 words.",
                                "Define concurrency in 8 words."], config={"max_concurrency": 3})
    for r in replies:
        print("•", r.text)
    print()
    async for chunk in llm.astream("Count from 1 to 5, words only."):
        print(chunk.text, end="", flush=True)

### What just happened?

- `gather` + `Semaphore(3)` and `abatch(max_concurrency=3)` both took ≈ 1 s for 6 calls: same result, one thread.
- `astream` is `stream` with `async for`. In Module 12, FastAPI endpoints are `async def` and use exactly these methods.
- **Never call blocking code inside `async def`** (`time.sleep`, `requests.get`, a sync `invoke`): it freezes every other task. Use the `a...` methods.

---
## Part 6 · `astream_events`: one stream, every step

Pipelines have several steps (prompt → model → parser; later agents with tools). `astream_events` reports **semantic events** from all of them: `on_chat_model_start`, `on_chat_model_stream` (each chunk), `on_chat_model_end` (full message), and similar events for other runnables ([Models → Streaming events](https://docs.langchain.com/oss/python/langchain/models#stream)). Filter by `event` and `name`.

In [ ]:
# 🧪 Offline: list the events of one fake streaming call.
fake = GenericFakeChatModel(messages=iter([AIMessage("Events are typed.")]))
async for event in fake.astream_events("hi", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(f"{kind:<22} chunk={event['data']['chunk'].text!r}")
    else:
        print(f"{kind:<22} name={event['name']}")

`version="v2"` is the current default for `astream_events` on runnables (the signature also accepts newer formats; check the [reference](https://reference.langchain.com/python/langchain-core/runnables/base/Runnable) before relying on them). **Agents and graphs** have their own `stream(..., stream_mode=...)` API, covered in Modules 02 and 07.

---
## Part 7 · Concurrency vs rate limits

Parallelism multiplies speed **and** usage. Groq limits you by requests and tokens per minute (verified 2026-10-11 in Lesson 00.07: about **8K tokens/min** for `gpt-oss` on the free tier).

| Calls in flight | Tokens per call | Tokens in ~1 s | Free-tier result |
|---|---|---|---|
| 2 | 400 | 800 | Fine |
| 10 | 800 | 8,000 | You hit the per-minute limit in about a second → 429s |

`max_concurrency` limits **how many at once**, not **how many per minute**. For that you need a rate limiter, and for 429s you need backoff that respects `retry-after`: both are Lesson 01.06.

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `RuntimeError: asyncio.run() cannot be called from a running event loop` | `asyncio.run` inside Jupyter | Use top-level `await` in notebooks; `asyncio.run` only in scripts |
| `SyntaxError: 'await' outside function` | Top-level `await` in a `.py` file, **or** in a Jupyter cell that starts with `%%time` | Scripts: `async def main()` + `asyncio.run(main())`. Notebooks: time with `time.perf_counter()` |
| Stream prints all at once at the end | Missing `flush=True` | `print(..., end="", flush=True)` |
| Output shows `content='...' additional_kwargs=...` | Printing the chunk object | Print `chunk.text` |
| Batch crashes and you lose all results | One input raised | `return_exceptions=True` |
| Async version isn't faster | Blocking code (`time.sleep`, sync `invoke`) inside `async def` | Use `asyncio.sleep`, `ainvoke`, `abatch` |
| Structured output + streaming fails on Groq | Not supported together | Stream text; use structured output without streaming |

## ⚠️ Production notes

> ⚠️ **Production note: unbounded parallelism.**
> `asyncio.gather` over a list with no semaphore fires everything at once. In production: always bound concurrency, add a client-side rate limiter (01.06), set per-request timeouts, and cancel remaining tasks when the caller gives up.

## 🏋️ Exercises

1. **(Easy)** In Part 4, try `max_concurrency` = 1, 2 and 6. Predict each wall time first, then check with `%%time`.
2. **(Easy)** Change `refresh_every` in `stream_to_markdown` to 1 and to 50 on the real model. What do you notice?
3. **(Medium)** Write `async def first_token_latency(model, prompt) -> float` that returns TTFT using `astream` and `time.perf_counter()`. Measure the fast and reasoning models 3 times each and print the medians.
4. **(Stretch)** Write `stream_with_timeout(model, prompt, seconds)` that stops reading a stream after `seconds` and returns what arrived so far. Hint: check the elapsed time inside the loop and `break`; breaking out of the iterator closes the HTTP connection.

<details>
<summary><b>Show solutions</b></summary>

**1.** ≈ 3.0 s, 1.5 s and 0.5 s: `ceil(6 / n) * 0.5`.

**3.**
```python
import statistics

async def first_token_latency(model, prompt: str) -> float:
    """Seconds until the first chunk arrives."""
    start = time.perf_counter()
    async for _ in model.astream(prompt):
        return time.perf_counter() - start
    return float("nan")

if HAVE_GROQ_KEY:
    for role in ("fast", "reasoning"):
        model = get_model(role, reasoning_effort="low")
        times = [await first_token_latency(model, "Say hi.") for _ in range(3)]
        print(role, f"median TTFT {statistics.median(times):.2f}s")
```

**4.**
```python
def stream_with_timeout(model, prompt: str, seconds: float) -> str:
    """Return the text streamed within `seconds`; stop reading after that."""
    start, parts = time.perf_counter(), []
    for chunk in model.stream(prompt):
        parts.append(chunk.text)
        if time.perf_counter() - start > seconds:
            break
    return "".join(parts)
```
</details>

## ✅ Checkpoint

- [ ] I can aggregate chunks with `+` and know what `chunk_position == "last"` means.
- [ ] I measured TTFT vs total time and can explain the difference.
- [ ] I rendered a live-updating Markdown answer with a display handle.
- [ ] I used `max_concurrency`, `return_exceptions=True` and `batch_as_completed`.
- [ ] I wrote top-level `await`, `gather` + `Semaphore`, `abatch` and `astream_events` code.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Does every course provider report token usage **while streaming**? You saw ChatGroq does. Find the answer for `ChatOpenAI` and `ChatGoogleGenerativeAI` in their [integration pages](https://docs.langchain.com/oss/python/integrations/chat/index) *and* their source (`stream_usage`). Log what the docs say vs what the code does in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Streams are iterators of `AIMessageChunk`; chunks add up to a full message, usage included.
- Streaming improves perceived latency (TTFT), not total time.
- `batch` = client-side threads, ordered results; bound it with `max_concurrency`, protect it with `return_exceptions=True`.
- Async = many waiting tasks on one thread. In Jupyter use top-level `await`; never block inside `async def`.
- Concurrency is not a rate limit. Next lesson: tokens, cost and rate limits.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 01.06 · Tokens, cost, and rate limits** → [`01-06-tokens-cost-rate-limits.ipynb`](01-06-tokens-cost-rate-limits.ipynb).

[Module 01 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)